## Calculate Scale and Zero Point

In [1]:
import numpy as np

In [2]:
# Tensor 1: Mixed range
t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)

# Tensor 2: All positive
t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)

# Tensor 3: All negative
t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32)

# Tensor 4: Constant tensor
t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32)

# Tensor 5: Very small values
t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32)

### Calculate Scale and Zero Point

In [3]:
def calculate_scale_zero_point(tensor, q_min=-128, q_max=127):
    """
    Calculate affine INT8 quantization parameters.

    scale = (x_max - x_min) / (q_max - q_min)
    zero_point = round(q_min - x_min/scale)

    Handles:
      - Constant tensors
      - Extremely small ranges
    """

    x_min = float(np.min(tensor))
    x_max = float(np.max(tensor))

    # Constant tensor
    if np.isclose(x_max, x_min):
        return 1.0, 0

    scale = (x_max - x_min) / (q_max - q_min)

    # Protect against extremely tiny ranges
    eps = 1e-12
    scale = max(scale, eps)

    zero_point = round(q_min - (x_min / scale))

    zero_point = int(
        np.clip(
            zero_point,
            q_min,
            q_max
        )
    )

    return scale, zero_point

### Quantization Function

In [4]:
def quantize_tensor(tensor, scale, zero_point,
                    q_min=-128, q_max=127):
    q = np.round(tensor / scale) + zero_point

    q = np.clip(
        q,
        q_min,
        q_max
    )

    return q.astype(np.int8)

### Dequantization Function

In [5]:
def dequantize_tensor(quantized_tensor,
                      scale,
                      zero_point):

    return (
        quantized_tensor.astype(np.float32)
        - zero_point
    ) * scale

### Evaluation Function

In [6]:
def evaluate_tensor(name, tensor):

    scale, zero_point = calculate_scale_zero_point(tensor)

    q = quantize_tensor(
        tensor,
        scale,
        zero_point
    )

    dq = dequantize_tensor(
        q,
        scale,
        zero_point
    )

    mae = np.mean(
        np.abs(tensor - dq)
    )

    print(f"\n{'='*60}")
    print(f"Tensor: {name}")
    print(f"{'='*60}")

    print("Tensor values:")
    print(tensor)

    print("\nTensor min:")
    print(np.min(tensor))

    print("\nTensor max:")
    print(np.max(tensor))

    print("\nScale:")
    print(scale)

    print("\nZero point:")
    print(zero_point)

    print("\nQuantized tensor:")
    print(q)

    print("\nDequantized tensor:")
    print(dq)

    print("\nMean Absolute Error:")
    print(mae)

### Run All Test Cases

In [7]:
evaluate_tensor("t1 - Mixed Range", t1)

evaluate_tensor("t2 - All Positive", t2)

evaluate_tensor("t3 - All Negative", t3)

evaluate_tensor("t4 - Constant Tensor", t4)

evaluate_tensor("t5 - Very Small Values", t5)


Tensor: t1 - Mixed Range
Tensor values:
[-1.5 -0.8  0.   0.9  2.3]

Tensor min:
-1.5

Tensor max:
2.3

Scale:
0.014901960597318761

Zero point:
-27

Quantized tensor:
[-128  -81  -27   33  127]

Dequantized tensor:
[-1.505098   -0.80470586  0.          0.8941176   2.2949018 ]

Mean Absolute Error:
0.004156864

Tensor: t2 - All Positive
Tensor values:
[0.1 0.5 1.2 2.  3.5]

Tensor min:
0.1

Tensor max:
3.5

Scale:
0.013333333327489741

Zero point:
-128

Quantized tensor:
[-120  -90  -38   22  127]

Dequantized tensor:
[0.10666667 0.50666666 1.2        2.         3.4       ]

Mean Absolute Error:
0.022666646

Tensor: t3 - All Negative
Tensor values:
[-3.  -2.1 -1.4 -0.6 -0.1]

Tensor min:
-3.0

Tensor max:
-0.1

Scale:
0.011372549013764251

Zero point:
127

Quantized tensor:
[-128  -58    4   74  118]

Dequantized tensor:
[-2.9        -2.1039217  -1.3988236  -0.6027451  -0.10235295]

Mean Absolute Error:
0.022039209

Tensor: t4 - Constant Tensor
Tensor values:
[5. 5. 5.]

Tensor min:
5.

### Observations

In [ ]:
print("""
OBSERVATIONS

Tensor 1 (Mixed Range):
- Contains negative, positive, and zero values.
- Zero point is typically close to zero.
- Quantization error is small.

Tensor 2 (All Positive):
- Entire range is above zero.
- Zero point shifts toward q_min (-128).
- Uses more INT8 range efficiently.

Tensor 3 (All Negative):
- Entire range is below zero.
- Zero point shifts toward q_max (127).
- Preserves negative values efficiently.

Tensor 4 (Constant Tensor):
- x_min == x_max.
- Normal scale computation would divide by zero.
- Scale is set to 1.0 and zero point to 0.

Tensor 5 (Very Small Values):
- Dynamic range is extremely small.
- Scale becomes very small.
- Quantization still preserves differences.
- Numerical precision becomes important.
""")